<a href="https://colab.research.google.com/github/sanyuktaraut09/Word2Vec-NLP-Project/blob/main/NLP_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Word Embedding & Sentiment Text Processing using Word2Vec

### Loading the Dataset
In this initial step, load the dataset.

In [ ]:
import pandas as pd

try:
    df = pd.read_csv('selective-stock-headlines-sentiment.csv', encoding='latin1')
    print(df.head())
except FileNotFoundError:
    print("File Not Found.")


**Observation & Inference:**
* **Observation:** The dataset is successfully loaded into a Pandas DataFrame.
* **Inference:** By observing the initial rows, we verify the data structure, confirming it contains the required text (`headlines`) and `sentiment` columns needed for downstream processing.

### Text Preprocessing
Clean the text data to prepare it for modeling. This involves converting all text to lowercase, alongside stripping away punctuation, special characters, and standard English stopwords.

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

# Define stopwords
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Convert text to lower case
    text = str(text).lower()
    # Remove punctuations and special characters
    text = re.sub(r'[^a-z\s]', '', text)
    # Remove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)

# The text column is named 'headline'
df['Tweets'] = df['headline'].apply(preprocess_text)

print(df[['headline', 'Tweets']].head())

# Justification & Inference:
# We lowercase the text to ensure uniformity (so 'Stock' and 'stock' are treated as the same word). Removing punctuation and stopwords reduces noise in our dataset, as these elements generally do not contribute to the underlying sentiment.

### Creating X and y Objects
Separate features (the text) from target variable (the sentiment).

In [ ]:
# Create two objects X and y. X will be the 'Tweets' column dataframe and y will be the “Sentiment” column.
X = df['Tweets']
y = df['sentiment']

print("X shape:", X.shape)
print("y shape:", y.shape)

# Observation:
# The dataset is successfully split into predictor (X) and target (y) variables, which is the standard format required for any subsequent machine learning pipeline.

### Vectorization (CBOW and Skip-Gram)
Used the gensim library to vectorize the text using both Continuous Bag of Words (CBOW) and Skip-Gram models. Also find the second most frequent word to display its embedding.

In [ ]:
from gensim.models import Word2Vec
from collections import Counter

# Tokenize sentences for Word2Vec
sentences = [tweet.split() for tweet in X]

# Find the second most frequent word
all_words = [word for sentence in sentences for word in sentence]
word_counts = Counter(all_words)
# most_common(2) returns a list of the top 2: [(word1, count1), (word2, count2)]
second_most_freq_word = word_counts.most_common(2)[1][0]
print(f"The second most frequent word is: '{second_most_freq_word}'")

# Vectorize using CBOW (sg=0)
cbow_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, sg=0)

# Vectorize using Skip-Gram (sg=1)
skipgram_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, sg=1)

# Display embeddings
cbow_embedding = cbow_model.wv[second_most_freq_word]
sg_embedding = skipgram_model.wv[second_most_freq_word]

print(f"\nCBOW Embedding for '{second_most_freq_word}':\n", cbow_embedding[:5], "... (truncated)")
print(f"\nSkip-Gram Embedding for '{second_most_freq_word}':\n", sg_embedding[:5], "... (truncated)")

# Justification:
# Word2Vec creates dense vector representations of words. CBOW predicts a target word based on context, while Skip-Gram predicts context based on a target word.

### Visualization of Embeddings
To visualize the 100-dimensional embeddings, plot them as a line/bar chart to see the variations in the vector dimensions.

In [ ]:
import matplotlib.pyplot as plt

# Display the two embeddings using a visualization
plt.figure(figsize=(12, 5))

plt.plot(cbow_embedding, label='CBOW', alpha=0.7, color='blue')
plt.plot(sg_embedding, label='Skip-Gram', alpha=0.7, color='orange')

plt.title(f"Embedding Vector Comparison for the word: '{second_most_freq_word}'")
plt.xlabel("Vector Dimension Index")
plt.ylabel("Vector Value")
plt.legend()
plt.grid(True)
plt.show()

# Observation & Inference on Embedding Techniques:
# 1. Observation: The plot shows that the CBOW and Skip-Gram models generate entirely different numeric vector representations for the exact same word.
# 2. Inference: CBOW smooths over context words by treating them as an aggregate (bag of words context), which generally trains faster and represents frequent words well.
# Skip-Gram treats each context word as a new observation, leading to better representations for rare words but requiring more training time.
# The difference in their vector space is due to these differing training objectives.

### 6: Count-based vs. Prediction-based Word Embeddings

* **Count-based Embeddings (e.g., Bag of Words, TF-IDF, Co-occurrence Matrices):**
    * **Mechanism:** These rely on counting the global frequency of words in documents or how often words co-occur in a corpus.
    * **Semantic Relationships:** Traditional BoW and TF-IDF are poor at capturing semantic relationships. They treat words as orthogonal (independent) features, meaning the vector for "king" is no closer to "queen" than it is to "apple".
    * **Sparsity:** They result in highly sparse, high-dimensional matrices (the size of the entire vocabulary), which consumes a lot of memory.

* **Prediction-based Embeddings (e.g., Word2Vec CBOW & Skip-gram):**
    * **Mechanism:** These are neural network-based models that learn word representations by trying to predict a word from its context (CBOW) or the context from a word (Skip-gram).
    * **Semantic Relationships:** They are highly effective at capturing rich semantic and syntactic relationships. Because words that appear in similar contexts are mapped to similar vector spaces, they can capture complex analogies (e.g., Vector("King") - Vector("Man") + Vector("Woman") ≈ Vector("Queen")).
    * **Sparsity:** They create dense, low-dimensional vectors (e.g., 100 to 300 dimensions), which are much more computationally efficient for downstream machine learning tasks.

**Inference:** Prediction-based models are vastly superior for capturing deep semantic meaning and context compared to pure count-based models, though count-based models are easier to interpret and faster to construct initially.